# Task 2: Supervised Learning
**Objective:** Engineer new features, perform 5-fold cross-validation, and train classification models.
**Inputs:** `../data/cleaned.csv`
**Outputs:** `../models/supervised_best.pkl`

In [7]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

df = pd.read_csv('../data/cleaned.csv')

df['Review_Word_Count'] = df['Translated_Review'].apply(lambda x: len(str(x).split()))
df['Review_Char_Count'] = df['Translated_Review'].apply(lambda x: len(str(x)))

X_text = df['Translated_Review'].astype(str)
y = df['Sentiment']

tfidf = TfidfVectorizer(max_features=500)
X_tfidf = tfidf.fit_transform(X_text)

scaler = StandardScaler()
X_num = scaler.fit_transform(df[['Review_Word_Count', 'Review_Char_Count']])

X = hstack((X_tfidf, X_num))

### Model Training and Cross-Validation
**Action:** Split data 80/20, apply 5-fold CV, and train models.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_lr = LogisticRegression(max_iter=1000)
cv_lr = cross_val_score(model_lr, X_train, y_train, cv=5)

model_dt = DecisionTreeClassifier(random_state=42)
cv_dt = cross_val_score(model_dt, X_train, y_train, cv=5)

print(f"Logistic Regression CV Accuracy: {cv_lr.mean():.4f}")
print(f"Decision Tree CV Accuracy: {cv_dt.mean():.4f}")

Logistic Regression CV Accuracy: 0.8512
Decision Tree CV Accuracy: 0.7723


### Evaluation and Model Export
**Action:** Evaluate on the test set and save the best performing model.

In [9]:
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)
print("Logistic Regression Test Results:")
print(classification_report(y_test, y_pred_lr))

model_dt.fit(X_train, y_train)
y_pred_dt = model_dt.predict(X_test)
print("Decision Tree Test Results:")
print(classification_report(y_test, y_pred_dt))

with open('../models/supervised_best.pkl', 'wb') as f:
    pickle.dump(model_lr, f)

Logistic Regression Test Results:
              precision    recall  f1-score   support

    Negative       0.82      0.65      0.73      1272
     Neutral       0.71      0.83      0.77       871
    Positive       0.90      0.93      0.91      3796

    accuracy                           0.85      5939
   macro avg       0.81      0.80      0.80      5939
weighted avg       0.85      0.85      0.85      5939

Decision Tree Test Results:
              precision    recall  f1-score   support

    Negative       0.59      0.58      0.59      1272
     Neutral       0.67      0.70      0.68       871
    Positive       0.86      0.86      0.86      3796

    accuracy                           0.77      5939
   macro avg       0.71      0.71      0.71      5939
weighted avg       0.77      0.77      0.77      5939

